# HW1 Submission 

**Before you start, copy this notebook into your own workspace folder.** 

Follow this template for Questions 1, 2, 3 and the evidence section of Question 4, filling in your code in the cells indicated. Feel free to add as many cells as you want, but keep the same layout and do not remove the section headings.

For Question 2, you may draft each query in the `%sql` cell provided, then run it with `save_query`, which writes the `.sql` and `.csv` files you submit.
The written parts of Question 4 are submitted as a PDF; only the code that produced your numbers goes here.

Split your work into several cells rather than writing one long one, and add a short comment where a step is not obvious from the code itself. Do not write an essay in comments: a line explaining why you filtered something, or what a parsing step returns, is enough.

Everything you submit must be what this notebook produced. The files are written by the helpers provided below, not by hand, and the numbers in your Question 4 report must come from code in the evidence section. Do not edit a saved file afterwards, and do not report a number the notebook cannot reproduce. We check this.

Before you submit: restart the notebook and run it from beginning to end. It must complete without errors, and every output must be visible in the exported HTML.

## Load the data

Run the two cells below to load the data.

### Load the DataFrame

Run the cell below to read the data into `airbnb_df`. Use this for Questions 1, 3 and 4.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType, StructType, StructField

DATA_PATH = "/Volumes/94290_lab/course_data/airbnb/airbnb_1_12_parquet"
airbnb_df = spark.read.parquet(DATA_PATH)
print(f"rows: {airbnb_df.count():,}   columns: {len(airbnb_df.columns)}")
display(airbnb_df.limit(5))

%md
### Load the SQL view

Run the cell below to make the same data available to the `%sql` cells as `airbnb_data`. Use this for Question 2.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW airbnb_data AS
SELECT * FROM parquet.`/Volumes/94290_lab/course_data/airbnb/airbnb_1_12_parquet`;
SELECT * FROM airbnb_data LIMIT 5


## Question 1: Data Handling

PySpark only. No Pandas, NumPy or similar libraries.

Each question produces one table, saved with the `save_csv` helper below. Run the next cell once before you start.

In [ ]:
import os

try:
    # inside a Databricks notebook this is the folder the notebook lives in
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext()\
        .notebookPath().get()
    OUT_DIR = "/Workspace" + notebook_path.rsplit("/", 1)[0]
except Exception:
    # Databricks Connect has no notebook path; dbutils raises "cluster_id is required"
    OUT_DIR = os.getcwd()
print(OUT_DIR)


def wait_for_file(path, timeout=30):
    """Wait until a written file is visible and non-empty. Do not modify."""
    import os, time
    start = time.time()
    while time.time() - start < timeout:
        if os.path.exists(path) and os.path.getsize(path) > 0:
            return path
        time.sleep(0.5)
    raise TimeoutError(f"{path} did not appear within {timeout}s")


def save_csv(sdf, file_name, out_dir=OUT_DIR):
    """Write a small result table to <out_dir>/<file_name>.csv. Do not modify this function."""
    import os, csv
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"{file_name}.csv")
    rows = sdf.collect()
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f, quoting=csv.QUOTE_MINIMAL)
        w.writerow(sdf.columns)
        for r in rows:
            w.writerow(["" if v is None else v for v in r])
        f.flush()
        os.fsync(f.fileno())
    wait_for_file(path)
    print(f"wrote {len(rows)} rows to {path}")
    return path


def save_query(query, file_name, out_dir=OUT_DIR):
    """Run a SQL query, save the query text as .sql and its result as .csv. Do not modify."""
    import os
    os.makedirs(out_dir, exist_ok=True)
    q = query.strip().rstrip(";")
    sql_path = os.path.join(out_dir, f"{file_name}.sql")
    with open(sql_path, "w", encoding="utf-8") as f:
        f.write(q + "\n")
        f.flush()
        os.fsync(f.fileno())
    wait_for_file(sql_path)
    result = spark.sql(q)
    save_csv(result, file_name)
    return result


def show_saved(file_name, out_dir=OUT_DIR):
    """Read back a saved CSV and display it. Do not modify."""
    import csv
    with open(f"{out_dir}/{file_name}.csv", newline="", encoding="utf-8") as f:
        rows = list(csv.reader(f))
    print(f"{len(rows) - 1} data rows")
    return spark.createDataFrame(rows[1:], rows[0])

%md
### Question 1a

The 15 properties with the largest gap between their highest and lowest category rating.

Your final DataFrame must be called `q1a`, with the columns already in the required order.

In [ ]:
cat_schema = ArrayType(StructType([
    StructField("name", StringType()),
    StructField("value", StringType()),
]))

q1a_base = (
    airbnb_df
    .withColumn("overall_rating", F.col("ratings").cast("double")) # col 3
    .withColumn("cats_raw", F.from_json("category_rating", cat_schema))
    .withColumn("cats", F.transform("cats_raw", lambda c: F.struct(
        c["name"].alias("name"),
        F.regexp_replace(c["value"], ",", ".").try_cast("double").alias("value"), # def 1
    )))
)

q1a_base.select("property_id", "overall_rating", "cats").printSchema()

In [ ]:
q1a_work = ( ### col 1, 2?
    q1a_base
    .withColumn("cats_present", F.filter("cats", lambda c: c["name"].isNotNull() & c["value"].isNotNull())) # def 1 ### numeric rating?
    .where((F.col("overall_rating") >= 4.0) & (F.col("overall_rating") < 5.0)) # filter 1
    .where(F.exists("cats_present", lambda c: c["name"] == "Cleanliness")) # filter 2
    .where(F.col("property_id").isNotNull()) # filter 3
    .withColumn("num_categories", F.size("cats_present")) # col 4
    .withColumn("best_category_value", F.array_max(F.transform("cats_present", lambda c: c["value"]))) # col 6
    .withColumn("worst_category_value", F.array_min(F.transform("cats_present", lambda c: c["value"]))) # col 8
    .withColumn("best_category", F.array_min(F.transform(
        F.filter("cats_present", lambda c: c["value"] == F.col("best_category_value")),
        lambda c: c["name"]))) # col 5, def 3
    .withColumn("worst_category", F.array_min(F.transform(
        F.filter("cats_present", lambda c: c["value"] == F.col("worst_category_value")),
        lambda c: c["name"]))) # col 7, def 3
    .withColumn("rating_gap", F.col("best_category_value") - F.col("worst_category_value")) # col 9, def 2
)

In [ ]:
q1a = (
    q1a_work
    .select(
        F.col("property_id").cast("string"),
        F.col("listing_name").cast("string"),
        F.col("overall_rating").cast("double"),
        F.col("num_categories").cast("int"),
        F.col("best_category").cast("string"),
        F.col("best_category_value").cast("double"),
        F.col("worst_category").cast("string"),
        F.col("worst_category_value").cast("double"),
        F.col("rating_gap").cast("double"),
    )
    .orderBy(
        F.desc("rating_gap"),
        F.desc("overall_rating"),
        F.asc("property_id"),
        F.desc("num_categories"),
        F.asc("best_category"),
    )
    .limit(15)
)

q1a.printSchema()
display(q1a)

In [0]:
save_csv(q1a, "q1a")
display(show_saved("q1a"))

### Question 1b

The 25 properties with the most beds that do not offer TV and allow check-in at 2:00 PM, among properties priced from 100 to 400 US dollars per night, both ends included, with 4 or more beds.

Your final DataFrame must be called `q1b`, with the columns already in the required order.

In [0]:
q1b_price = airbnb_df.withColumn(
    "price_usd",
    F.when(F.col("currency") == "USD", F.col("price").cast("double"))
     .when(F.col("currency") == "EUR", F.col("price").cast("double") * 1.08)
) # def 4, col 3

In [ ]:
amen_item = StructType([StructField("name", StringType()), StructField("value", StringType())])
amen_schema = ArrayType(StructType([
    StructField("group_name", StringType()),
    StructField("items", ArrayType(amen_item)),
])) # def 2

NOT_INCLUDED_GROUPS = ["Not included", "Nicht enthalten", "Non inclus"]
MISSING_VALUE_RE = (
    r"^SYSTEM_NO_|^SYSTEM_OFFLINE$"
    r"|^There is no .* on the (property|bedroom door)\.$|^This place may not have a |^Host has indicated (that )?no "
    r"|^Es gibt kein|gibt es möglicherweise kein|hat angegeben, dass kein"
) # def 1


def is_available(group, item):
    missing = (F.coalesce(group["group_name"].isin(NOT_INCLUDED_GROUPS), F.lit(False))
               | F.coalesce(item["value"].rlike(MISSING_VALUE_RE), F.lit(False)))
    return ~missing


q1b_amen = (
    q1b_price
    .withColumn("amen_groups", F.from_json("amenities", amen_schema))
    # flatten groups into one list of items per row, tagging each item as available or not
    .withColumn("amen_items", F.flatten(F.transform("amen_groups", lambda g: F.transform(
        g["items"], lambda i: F.struct(
            i["name"].alias("name"),
            i["value"].alias("value"),
            is_available(g, i).alias("available"),
        )))))
    .withColumn("amen_available", F.filter("amen_items", lambda i: i["available"])) # def 1
    .withColumn("num_amenities", F.coalesce(F.size("amen_available"), F.lit(0))) # def 2, col 5
    .withColumn("has_tv", F.coalesce(F.exists("amen_available", lambda i: i["value"] == "SYSTEM_TV"), F.lit(False))) # filter 2
)

In [ ]:
BEDS_RE = r"^(\d+) ([A-Za-z ]+ )?beds?$"

q1b_beds = (# def 3
    q1b_amen
    .withColumn("details_arr", F.from_json("details", ArrayType(StringType())))
    .withColumn("beds_entry", F.get(F.filter("details_arr", lambda d: d.rlike(BEDS_RE)), 0))
    .withColumn("num_beds", F.regexp_extract("beds_entry", BEDS_RE, 1).try_cast("int")) # col 4 
)

In [ ]:
# def 5
def normalise_rule(c): 
    """For parsing only: narrow/no-break spaces -> space, lower case, "p.m." -> "pm"."""
    c = F.regexp_replace(c, "[\u202f\u00a0]", " ")
    return F.regexp_replace(F.lower(c), r"([ap])\.m\.", "$1m")


TIME = r"(\d{1,2}):(\d{2}) ?(am|pm)?"
AFTER_RE = rf"^check-in after {TIME}$"
RANGE_RE = rf"^check-in: {TIME} ?[-–] ?{TIME}$"
FLEX_RE = r"^flexible check-in$"
CHECKIN_RE = f"{AFTER_RE}|{RANGE_RE}|{FLEX_RE}"


def to_minutes(rule, regex, first_group):
    """Minutes after midnight from the hour, minute and am/pm groups (24-hour clock when am/pm is missing)."""
    hour = F.regexp_extract(rule, regex, first_group).try_cast("int")
    minute = F.regexp_extract(rule, regex, first_group + 1).try_cast("int")
    ampm = F.regexp_extract(rule, regex, first_group + 2)
    hour = F.when(ampm == "pm", hour % 12 + 12).when(ampm == "am", hour % 12).otherwise(hour)
    return hour * 60 + minute


TWO_PM = 14 * 60

q1b_checkin = (
    q1b_beds
    .withColumn("rules_arr", F.from_json("house_rules", ArrayType(StringType())))
    .withColumn("check_in_rule", F.get(F.filter("rules_arr", lambda r: normalise_rule(r).rlike(CHECKIN_RE)), 0)) # col 6
    .withColumn("ci_norm", normalise_rule(F.col("check_in_rule")))
    .withColumn("ci_start", F.coalesce(to_minutes(F.col("ci_norm"), AFTER_RE, 1),
                                       to_minutes(F.col("ci_norm"), RANGE_RE, 1)))
    .withColumn("ci_end", to_minutes(F.col("ci_norm"), RANGE_RE, 4))
    .withColumn(
        "allows_2pm",
        F.when(F.col("ci_norm").rlike(FLEX_RE), True)
         .when(F.col("ci_end").isNull(), F.col("ci_start") <= TWO_PM)
         .when(F.col("ci_end") > F.col("ci_start"),
               (F.col("ci_start") <= TWO_PM) & (F.col("ci_end") >= TWO_PM))
         .otherwise((F.col("ci_start") <= TWO_PM) | (F.col("ci_end") >= TWO_PM))
    )
)

In [ ]:
# null price_usd / num_beds / allows_2pm fail these comparisons, so those rows are excluded
q1b = (
    q1b_checkin
    .where(F.col("price_usd").between(100, 400)) # filter 1
    .where(~F.col("has_tv")) # filter 2
    .where(F.col("allows_2pm")) # filter 3 
    .where(F.col("num_beds") >= 4) # filter 4
    .select(
        F.col("property_id").cast("string"), 
        F.col("listing_name").cast("string"),
        F.col("price_usd").cast("double"),
        F.col("num_beds").cast("int"),
        F.col("num_amenities").cast("int"),
        F.col("check_in_rule").cast("string"),
    )
    .orderBy(
        F.desc("num_beds"),
        F.asc("price_usd"),
        F.desc("num_amenities"),
        F.asc("property_id"),
        F.asc("listing_name"),
        F.asc("check_in_rule"),
    )
    .limit(25)
)

q1b.printSchema()
display(q1b)

In [0]:
save_csv(q1b, "q1b")

# read the saved file back and show it, so what you see is what will be submitted
display(show_saved("q1b"))

## Question 2: SQL Queries

Write each query as a string in the cell provided, then run it with `save_query`, which runs the query, saves the text as `q2a.sql` or `q2b.sql`, and saves the result as `q2a.csv` or `q2b.csv`.

Each query must be a single statement that runs on its own and returns exactly the columns listed, with those names, in that order. Use `WITH` clauses for intermediate steps.

You can develop your query in a `%sql` cell first if you prefer the editor, then paste the finished query into the string below. If `airbnb_data` is not found, re-run the setup cell at the top of the notebook.

### Question 2a

The 10 hosts with the highest average rating across their great value listings.


In [0]:
%sql
-- YOUR QUERY HERE

In [0]:
q2a_query = """
 YOUR QUERY HERE 
"""

save_query(q2a_query, "q2a")

# read the saved file back and show it, so what you see is what will be submitted
display(show_saved("q2a"))

### Question 2b

The 10 superhosts whose listings are the most expensive on average.



In [0]:
%sql
-- YOUR QUERY HERE

In [0]:
q2b_query = """
 YOUR QUERY HERE
"""

save_query(q2b_query, "q2b")

# read the saved file back and show it, so what you see is what will be submitted
display(show_saved("q2b"))

## Question 3: Data Exploration & Visualization

Exploration and filtering in PySpark only. Charts may use matplotlib, seaborn, plotly or a dashboarding tool; if you use an external tool, the code that produced the aggregated data must still appear here.

At most 2 charts per section. A figure made of several related subplots counts as one chart, as long as the subplots answer the same question together, for example the same comparison across a few categories, or one distribution split by group. Say in your explanation why the subplots belong together. Unrelated subplots packed into one figure count separately.

For each chart, state the filtering or sampling you applied and how many rows were left, and keep your explanation to at most 3 sentences.

### Question 3a

How do the textual reviews of highly rated properties differ from those of poorly rated ones?

In [0]:
# YOUR CODE HERE

**Assumptions and filtering (3a):**

- What counts as highly rated / poorly rated:
- Filtering or sampling applied, and rows remaining:
- Size of each group:

**What the chart shows:**

### Question 3b

How does the set of amenities offered differ between two groups of listings types that you choose to compare?

In [0]:
# YOUR CODE HERE

**Assumptions and filtering (3b):**

- The two groups being compared, and why they are worth comparing:
- Filtering or sampling applied, and rows remaining:
- Size of each group:

**What the chart shows:**

### Question 3c

Your own question, answered with a visualization.

**Our question:**

**Why it is worth asking:**

In [0]:
# YOUR CODE HERE

**Assumptions and filtering (3c):**

- Filtering or sampling applied, and rows remaining:

**What the chart shows:**

## Question 4 Evidence

The written report goes in `q4.pdf`. This section holds only the code that produced the numbers and examples you cite there. Every number in your report must be reproducible from a cell below.

### 4a, problem 1

**Problem:** (one line, matching the heading in your PDF)

In [0]:
# YOUR CODE HERE

### 4a, problem 2

**Problem:**

In [0]:
# YOUR CODE HERE

### 4a, problem 3

**Problem:**

In [0]:
# YOUR CODE HERE

### 4b: simple supporting evidence for your research question

**Research question:**

In [0]:
# YOUR CODE HERE